## Sentiment Analysis

### Load Data

In [1]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

# Read raw data
master_dir = os.path.dirname(os.getcwd())

processed_reviews = os.path.join(master_dir, "data", "processed_data", "feature_engineered_reviews.csv")

df_processed = pd.read_csv(processed_reviews)

df_processed.head()

,Review_ID,Review,Rating,Lemmatized_Tokens,Review_lemmatized,Platform,Restaurant,Sentiment_Label
0,R000001,came here for the high tea. great service espe...,4.0,"['come', 'high', 'tea', '.', 'great', 'service...",come high tea . great service especially mr. j...,Google,Cuisines Restaurant,positive
1,R000002,"5 stars for the service, even though some of t...",2.0,"['5', 'star', 'service', ',', 'even', 'though'...","5 star service , even though staff need train ...",Google,Cuisines Restaurant,negative
2,R000003,"hi, thank you for your service. but! i feel so...",1.0,"['hi', ',', 'thank', 'service', '.', 'but', '!...","hi , thank service . but ! feel so sorry food ...",Google,Cuisines Restaurant,negative
3,R000004,i have the worse buffer dinner ever so far. th...,1.0,"['bad', 'buffer', 'dinner', 'ever', 'so', 'far...",bad buffer dinner ever so far . spread so smal...,Google,Cuisines Restaurant,negative
4,R000005,"that is are known 5 elmark "" 9h72 "" & kdk "" 3 ...",5.0,"['know', '5', 'elmark', '``', '9h72', '``', '&...",know 5 elmark `` 9h72 `` & kdk `` 3 k14y9 & 1 ...,Google,Cuisines Restaurant,positive


### BERT

I using pre-trained transformer model to inference the result because training a transformer model takes many days. It is eassy for me to just using pre-trained transformer model that only takes 30 minutes to predict the result.

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from tqdm import tqdm

# 1. Setup GPU device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # Must print 'cuda', NOT 'cpu'

ModuleNotFoundError: No module named 'torch'

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

In [ ]:
# 2. Load Model & Tokenizer with FP16 Precision
MODEL_NAME = "terrencewee12/xlm-roberta-base-sentiment-multilingual-finetuned-v3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    torch_dtype=torch.float16
).to(device)
model.eval()

In [ ]:
# 3. Using the cleaned but not yet tokenized reviews, we will convert them to a list of strings for batch processing
reviews = df_processed["Review"].astype(str).tolist()

In [ ]:
# 4. Batch prediction function returning both Labels and Scores
def predict_sentiment(text_list, batch_size=64, max_len=256):
    labels = []
    scores = []
    
    # ID to string mapping from model configuration
    id2label = model.config.id2label
    
    with torch.no_grad():
        for i in tqdm(range(0, len(text_list), batch_size), desc="Inferring"):
            batch_texts = text_list[i : i + batch_size]
            
            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            
            # Apply softmax to get probability scores (0.0 to 1.0)
            probabilities = torch.softmax(outputs.logits, dim=-1)
            
            # Extract max probability and corresponding class ID
            max_scores, class_ids = torch.max(probabilities, dim=-1)
            
            # Map predictions to label strings and float scores
            for cid, score in zip(class_ids.cpu().tolist(), max_scores.cpu().tolist()):
                labels.append(id2label[cid].capitalize())
                scores.append(round(score, 4))
                
    return labels, scores
          

In [ ]:
# Execute inference
bert_labels, bert_scores = predict_sentiment(reviews, batch_size=64, max_len=256)

In [ ]:
# 5. Assign results to DataFrame
df_processed["BERT_Sentiment"] = bert_labels
df_processed["BERT_Score"] = bert_scores

In [ ]:
df_processed[["Review","Rating","BERT_Sentiment","BERT_Score"]].head(10)

In [ ]:
# Get the counts of each sentiment label
sentiment_counts = df_processed["BERT_Sentiment"].value_counts()

print(sentiment_counts)

In [ ]:
import matplotlib.pyplot as plt

df_processed["BERT_Sentiment"].value_counts().plot(
    kind="bar"
)

plt.title("BERT Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.show()

In [ ]:
pd.crosstab(
    df_processed["Sentiment_Label"],
    df_processed["BERT_Sentiment"],
    margins=True
)

In [ ]:
# Calc the agreement between the original sentiment labels and BERT predictions
agreement = (
    df_processed["Sentiment_Label"].str.lower() ==
    df_processed["BERT_Sentiment"].str.lower()
)

print(
    "Agreement:",
    agreement.mean() * 100,
    "%"
)

In [ ]:
# Extract the disagreements reviews between the original sentiment labels and BERT predictions
disagreements = df_processed[
    df_processed["Sentiment_Label"].str.lower() != df_processed["BERT_Sentiment"].str.lower()
].copy()

print("Number of disagreements:", len(disagreements))

In [ ]:
# Display the disagreements reviews
with pd.option_context('display.max_colwidth', None):
    display(disagreements[
        [
            "Review",
            "Rating",
            "Sentiment_Label",
            "BERT_Sentiment",
            "BERT_Score"
        ]
    ].head(10))

In [ ]:
# Extract low-confidence predictions (BERT_Score < 0.60)
low_confidence = df_processed[
    df_processed["BERT_Score"] < 0.60
].copy()

print("Low-confidence predictions:", len(low_confidence))

In [ ]:
# see what review make teh BERT model have low confidence
low_confidence[
    [
        "Review",
        "Rating",
        "BERT_Sentiment",
        "BERT_Score"
    ]
].head(20)

with pd.option_context('display.max_colwidth', None):
    display(low_confidence[
        [
            "Review",
            "Rating",
            "BERT_Sentiment",
            "BERT_Score"
        ]
    ].head(10))

In [ ]:
def confidence_category(score):
    if score >= 0.80:
        return "High"
    elif score >= 0.60:
        return "Medium"
    else:
        return "Low"

df_processed["BERT_Confidence"] = df_processed["BERT_Score"].apply(
    confidence_category
)

In [ ]:
df_processed["BERT_Confidence"].value_counts()

In [ ]:
df_processed.head(10)

### Lexicons

testing